OK WHAT WE'RE DOING IN THIS NOTEBOOK:

1) Combined geojson -> only the lines
2) lines df to postings df -> each row associated with a posting id
3) postings df to  full postings df -> all the info for each posting from the sweeps_df added to the postings df
4) postings df to polydots -> each row is given polydots for where the map will load to
5) polydots to clickable polydots -> each row is given a poly dot to where they'll go when clicked

THEN I"M GONNA GO TEST THAT I CAN RUN THIS ON HTML AND CSS AND THEN I'M GOING TO COME BACK FOR THE POLYGONS

In [28]:
# import statements

import json
import re
import ast
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, LineString, shape
import shapely
import math
import geopandas as gpd
import pandas as pd
from shapely.ops import substring, unary_union, linemerge
from shapely.geometry import Point, MultiLineString
import shapely

In [29]:
#load in the data

# Load raw files
with open('../data/combined_sweeps.geojson') as f:
    raw_geojson = json.load(f)

sweep_df = pd.read_csv('../data/sweep_df.csv')
dots_gdf = gpd.read_file('../data/posting_dots.geojson')

print(f"Raw geojson features: {len(raw_geojson['features'])}")
print(f"sweep_df rows: {len(sweep_df)}")
print(f"dots: {len(dots_gdf)}")

Raw geojson features: 796
sweep_df rows: 2206
dots: 1724


In [30]:
#combined geojson - > lines 
line_features = [
    f for f in raw_geojson['features']
    if f['geometry']['type'] == 'LineString'
]

lines_geojson = {
    "type": "FeatureCollection",
    "features": line_features
}

lines_gdf = gpd.GeoDataFrame.from_features(line_features, crs="EPSG:4326")

print(f"Line features: {len(lines_gdf)}")
print(lines_gdf[['location', 'num_postings', 'sweep_event_ids']].head())

Line features: 627
                                            location  num_postings  \
0  Knight Street between Douglas Avenue and Grame...             1   
1       105th Ave between Russet St & San Leandro St             1   
2  Railroad Ave between 85th Ave and Louisana St ...             3   
3            E 12th St between 22nd Ave and 23rd Ave             1   
4  Lake Merritt: Amphitheater/ 12th St between La...             1   

      sweep_event_ids  
0               [0_0]  
1              [2_12]  
2  [2_14, 2_13, 2_12]  
3             [27_35]  
4             [31_17]  


In [31]:
#postings df to  full postings df -> all the info for each posting from the sweeps_df added to the postings df

def parse_array_field(val):
    """Handle both proper JSON arrays and Python string representations."""
    if isinstance(val, list):
        return val
    if isinstance(val, str):
        # Remove newlines, normalize spaces
        val = re.sub(r'\s+', ' ', val.strip())
        try:
            # Try standard JSON first
            return json.loads(val)
        except:
            pass
        try:
            # Try Python literal (handles numpy-style strings)
            return ast.literal_eval(val)
        except:
            pass
        # Last resort: split on spaces if it looks like "[51 111 877]"
        val = val.strip("[]").strip()
        return [v.strip().strip("'\"") for v in val.split() if v.strip()]
    return [val]

# Explode: one row per posting
rows = []
for _, feat in lines_gdf.iterrows():
    posting_ids = parse_array_field(feat['posting_ids'])
    op_dates    = parse_array_field(feat['operation_dates'])
    sweep_ids   = parse_array_field(feat['sweep_event_ids'])

    for i, pid in enumerate(posting_ids):
        rows.append({
            'location':          feat['location'],
            'posting_id':        int(pid),
            'dot_index':         i,
            'operation_date':    op_dates[i] if i < len(op_dates) else None,
            'sweep_event_id':    sweep_ids[i] if i < len(sweep_ids) else sweep_ids[0],
            'district':          feat['district'],
            'sensitivity_zone':  feat['sensitivity_zone'],
            'intervention_types':feat['intervention_types'],
            'num_postings':      feat['num_postings'],
            'geometry':          feat['geometry'],
        })

postings_from_lines = pd.DataFrame(rows)
print(f"Exploded rows: {len(postings_from_lines)}")
print(postings_from_lines[['location','posting_id','dot_index','operation_date']].head(8))

Exploded rows: 1724
                                            location  posting_id  dot_index  \
0  Knight Street between Douglas Avenue and Grame...        2046          0   
1       105th Ave between Russet St & San Leandro St         879          0   
2  Railroad Ave between 85th Ave and Louisana St ...          51          0   
3  Railroad Ave between 85th Ave and Louisana St ...         111          1   
4  Railroad Ave between 85th Ave and Louisana St ...         877          2   
5            E 12th St between 22nd Ave and 23rd Ave         546          0   
6  Lake Merritt: Amphitheater/ 12th St between La...        1241          0   
7              Poplar St between 20th St and 21st St         868          0   

                                      operation_date  
0                  October 12 2021 - October 12 2021  
1                        April 2 2025 - April 4 2025  
2  October 27 2025 - October 31 2025October 13 20...  
3                                               

In [32]:
#postings df to  full postings df -> all the info for each posting from the sweeps_df added to the postings df
# Columns to bring in from sweep_df (avoid duplicating ones we already have)
sweep_cols = [
    'posting_id', 'unique_location_id',
    'operation_start_date', 'operation_end_date',
    'intervention', 'before_after_grants_pass',
    'mid_lat', 'mid_lng'
]

sweep_subset = sweep_df[sweep_cols].drop_duplicates('posting_id')

postings_df = postings_from_lines.merge(
    sweep_subset,
    on='posting_id',
    how='left'
)

# Check join quality
matched   = postings_df['unique_location_id'].notna().sum()
unmatched = postings_df['unique_location_id'].isna().sum()
print(f"Matched: {matched} / Unmatched: {unmatched}")

# For unmatched (the 31 locations with name mismatches),
# fall back to dot_index-based location string
if unmatched > 0:
    print("Unmatched locations:")
    print(postings_df[postings_df['unique_location_id'].isna()]['location'].unique())

Matched: 1724 / Unmatched: 0


In [33]:
before = len(postings_df)
postings_df = postings_df.drop_duplicates(subset='posting_id', keep='first')
after = len(postings_df)
print(f"Dropped {before - after} rows")
print(f"Remaining rows: {after}")
print(f"Unique posting_ids: {postings_df['posting_id'].nunique()}")

Dropped 0 rows
Remaining rows: 1724
Unique posting_ids: 1724


Adding points   

In [ ]:
# ── params (keep whatever you had) ──────────────────────────────────────────
OFFSET_DIST = 0.0001   # lateral spacing between the two rows
DOT_SPACING  = 0.00007  # fixed distance between dots along the line (degrees)
START_OFFSET = 0.00008  # padding before first dot at the start of the line

In [35]:
from shapely.ops import unary_union, linemerge, substring
from shapely.geometry import MultiLineString
import shapely

def clean_offset(geom, dist, side):
    offset = geom.parallel_offset(dist, side, join_style=2)
    offset = unary_union(offset)
    
    if isinstance(offset, MultiLineString):
        offset = linemerge(offset)
    if isinstance(offset, MultiLineString):
        offset = max(offset.geoms, key=lambda x: x.length)
    
    if side == 'right':
        offset = shapely.geometry.LineString(list(offset.coords)[::-1])
    
    offset = shapely.force_2d(offset)
    
    if offset is None or offset.is_empty or offset.length < 1e-6 or len(list(offset.coords)) < 2:
        return None

    offset = substring(offset, 0.1, 0.9, normalized=True)
    return offset

In [36]:
# Each row has: location, posting_id, dot_index (sequential across all postings
# at that location), operation_start_date, sweep_event_id, unique_location_id, etc.

dot_rows = []

for location, group in postings_df.groupby('location'):
    # Get the line geometry for this location
    line_match = lines_gdf[lines_gdf['location'] == location]
    if len(line_match) == 0:
        continue
    geom = line_match.iloc[0]['geometry']
    n    = len(group)  # number of postings at this location

    # ── same geometry logic as before ───────────────────────────────────────
    mid_line     = shapely.force_2d(substring(geom, 0.1, 0.9, normalized=True))
    left_offset  = clean_offset(geom, OFFSET_DIST, 'left')
    right_offset = clean_offset(geom, OFFSET_DIST, 'right')

    def line_capacity(line):
        if line is None or line.is_empty:
            return 0
        return max(1, int((line.length - START_OFFSET) / DOT_SPACING) + 1)

    mid_cap   = line_capacity(mid_line)
    left_cap  = line_capacity(left_offset)
    right_cap = line_capacity(right_offset)

    mid_count   = min(n, mid_cap)
    left_count  = min(n - mid_count, left_cap)
    right_count = min(n - mid_count - left_count, right_cap)

    # ── assign posting rows to slots ────────────────────────────────────────
    # Sort postings by date so earliest = first dot placed
    sorted_postings = group.sort_values('operation_start_date').reset_index(drop=True)
    posting_idx = 0  # walks through sorted_postings

    for side, line, count in [
        ('middle', mid_line,    mid_count),
        ('left',   left_offset, left_count),
        ('right',  right_offset, right_count),
    ]:
        if count == 0 or line is None or line.is_empty:
            continue
        for j in range(count):
            dist_along = START_OFFSET + j * DOT_SPACING
            if dist_along > line.length:
                break
            if posting_idx >= len(sorted_postings):
                break

            pt      = line.interpolate(dist_along, normalized=False)
            posting = sorted_postings.iloc[posting_idx]

            dot_rows.append({
                # geometry
                'geometry':               pt,
                'load_lon':               pt.x,
                'load_lat':               pt.y,
                'side':                   side,
                'slot_index':             j,
                # posting identity
                'posting_id':             posting['posting_id'],
                'unique_location_id':     posting.get('unique_location_id'),
                'location':               location,
                # posting attributes
                'operation_start_date':   posting.get('operation_start_date'),
                'operation_end_date':     posting.get('operation_end_date'),
                'intervention':           posting.get('intervention'),
                'intervention_types':     posting.get('intervention_types'),
                'sweep_event_id':         posting.get('sweep_event_id'),
                'district':               posting.get('district'),
                'sensitivity_zone':       posting.get('sensitivity_zone'),
                'before_after_grants_pass': posting.get('before_after_grants_pass'),
                'num_postings':           n,
            })
            posting_idx += 1

dots_gdf = gpd.GeoDataFrame(dot_rows, geometry='geometry', crs=lines_gdf.crs)
print(f"Generated {len(dots_gdf)} dots across {len(lines_gdf)} segments")

# Sanity check
unassigned = n - posting_idx  # should be 0 for all locations
print(f"Postings with no dot slot: check locations where num_postings > capacity")

Generated 1724 dots across 627 segments
Postings with no dot slot: check locations where num_postings > capacity


In [37]:
print("Dots generated:", len(dots_gdf))
print("Rows in postings_df:", len(postings_df))
print("Unique posting_ids in postings_df:", postings_df['posting_id'].nunique())
print("Unique locations in postings_df:", postings_df['location'].nunique())
print("Unique locations in lines_gdf:", lines_gdf['location'].nunique())
print()

# Check if any location appears multiple times in lines_gdf
dupe_locs = lines_gdf['location'].value_counts()
print("Locations appearing >1x in lines_gdf:")
print(dupe_locs[dupe_locs > 1].head(10))

Dots generated: 1724
Rows in postings_df: 1724
Unique posting_ids in postings_df: 1724
Unique locations in postings_df: 627
Unique locations in lines_gdf: 627

Locations appearing >1x in lines_gdf:
Series([], Name: count, dtype: int64)


Add clickable locations

In [39]:
# Build bearing lookup from line geometries
line_bearings = {}
for _, feat in lines_gdf.iterrows():
    coords = list(feat['geometry'].coords)
    start, end = coords[0], coords[-1]
    dx = end[0] - start[0]
    dy = end[1] - start[1]
    line_bearings[feat['location']] = math.degrees(math.atan2(dx, dy))

def offset_point(lon, lat, bearing_deg, distance_km):
    perp = math.radians(bearing_deg + 90)
    R = 6371.0
    d = distance_km / R
    lat_r = math.radians(lat)
    lon_r = math.radians(lon)
    new_lat = math.asin(
        math.sin(lat_r) * math.cos(d) +
        math.cos(lat_r) * math.sin(d) * math.cos(perp)
    )
    new_lon = lon_r + math.atan2(
        math.sin(perp) * math.sin(d) * math.cos(lat_r),
        math.cos(d) - math.sin(lat_r) * math.sin(new_lat)
    )
    return math.degrees(new_lon), math.degrees(new_lat)

CLICK_OFFSET_KM = 0.018  # ~18m — tune this visually

clicked_lons, clicked_lats = [], []
for _, row in dots_gdf.iterrows():
    bearing = line_bearings.get(row['location'], 0)
    clon, clat = offset_point(row['load_lon'], row['load_lat'], bearing, CLICK_OFFSET_KM)
    clicked_lons.append(clon)
    clicked_lats.append(clat)

dots_gdf['clicked_lon'] = clicked_lons
dots_gdf['clicked_lat'] = clicked_lats

In [40]:
dots_gdf.to_file('posting_dots_4_2.geojson', driver='GeoJSON')